# 03 - ETL Parquet, PostgreSQL, dan Validasi
Parquet adalah output utama yang dapat dijalankan tanpa server. Load PostgreSQL tetap nonaktif sampai kredensial lab tersedia.

In [ ]:
from pathlib import Path
import os, time
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName('UMKM_ETL').master('local[*]')
         .config('spark.sql.shuffle.partitions','8')
         .config('spark.jars.packages','org.postgresql:postgresql:42.7.7')
         .getOrCreate())
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = ROOT / 'data' / 'synthetic_umkm_data.csv'
PARQUET_PATH = ROOT / 'output' / 'parquet' / 'umkm_spark'
if not CSV_PATH.exists(): CSV_PATH = Path('/content/synthetic_umkm_data.csv'); PARQUET_PATH = Path('/content/output/parquet/umkm_spark')

In [ ]:
# Baca sebagai string, lalu pilih dan cast secara eksplisit agar header khusus (%) tetap aman
raw = spark.read.option('header', True).option('inferSchema', False).csv(str(CSV_PATH))
df = raw.select(
    F.col('ID').cast('long').alias('umkm_id'),
    F.col('Monthly_Revenue').cast('double').alias('monthly_revenue'),
    F.col('Net_Profit_Margin (%)').cast('double').alias('net_profit_margin_pct'),
    F.col('Burn_Rate_Ratio').cast('double').alias('burn_rate_ratio'),
    F.col('Transaction_Count').cast('int').alias('transaction_count'),
    F.col('Avg_Historical_Rating').cast('double').alias('avg_historical_rating'),
    F.col('Review_Text').alias('review_text'),
    F.col('Review_Volatility').cast('double').alias('review_volatility'),
    F.col('Business_Tenure_Months').cast('int').alias('business_tenure_months'),
    F.col('Repeat_Order_Rate (%)').cast('double').alias('repeat_order_rate_pct'),
    F.col('Digital_Adoption_Score').cast('double').alias('digital_adoption_score'),
    F.col('Peak_Hour_Latency').alias('peak_hour_latency'),
    F.col('Location_Competitiveness').cast('int').alias('location_competitiveness'),
    F.col('Sentiment_Score').cast('double').alias('sentiment_score'),
    F.col('Class').alias('business_class'))
df = (df.withColumn('estimated_monthly_profit', F.round(F.col('monthly_revenue')*F.col('net_profit_margin_pct')/100,2))
      .withColumn('estimated_monthly_cost', F.round(F.col('monthly_revenue')-F.col('estimated_monthly_profit'),2))
      .withColumn('revenue_per_transaction', F.round(F.col('monthly_revenue')/F.col('transaction_count'),2))
      .withColumn('tenure_years', F.round(F.col('business_tenure_months')/12,2))
      .withColumn('tenure_band', F.when(F.col('business_tenure_months')<24,'<2 tahun').when(F.col('business_tenure_months')<60,'2-5 tahun').when(F.col('business_tenure_months')<120,'5-10 tahun').otherwise('>=10 tahun'))
      .withColumn('digital_adoption_band', F.when(F.col('digital_adoption_score')<3,'Rendah').when(F.col('digital_adoption_score')<6,'Menengah').when(F.col('digital_adoption_score')<8,'Tinggi').otherwise('Lanjutan'))
      .withColumn('sentiment_band', F.when(F.col('sentiment_score')<0,'Negatif').when(F.col('sentiment_score')>0,'Positif').otherwise('Netral'))
      .withColumn('operational_risk', F.when((F.col('net_profit_margin_pct')<0)&(F.col('burn_rate_ratio')>1),'High').when((F.col('net_profit_margin_pct')<0)|(F.col('burn_rate_ratio')>1),'Medium').otherwise('Low')))
input_rows = df.count(); print('Input rows:', input_rows)

In [ ]:
# Quality gate sebelum write
duplicates = df.groupBy('umkm_id').count().filter(F.col('count') > 1).count()
invalid = df.filter(F.col('umkm_id').isNull() | (F.col('monthly_revenue') <= 0) | (F.col('transaction_count') <= 0) | (~F.col('avg_historical_rating').between(1,5)) | (~F.col('repeat_order_rate_pct').between(0,100)) | (~F.col('digital_adoption_score').between(0,10))).count()
assert input_rows >= 100000 and duplicates == 0 and invalid == 0
print({'input_rows': input_rows, 'duplicate_ids': duplicates, 'invalid_rows': invalid})

In [ ]:
# Tulis Parquet Snappy dan baca ulang
started = time.perf_counter()
df.write.mode('overwrite').option('compression','snappy').parquet(str(PARQUET_PATH))
write_seconds = time.perf_counter() - started
readback = spark.read.parquet(str(PARQUET_PATH))
assert readback.count() == input_rows
assert readback.groupBy('umkm_id').count().filter(F.col('count') > 1).count() == 0
print({'parquet_rows': readback.count(), 'write_seconds': round(write_seconds,2), 'path': str(PARQUET_PATH)})

In [ ]:
# Load PostgreSQL hanya jika diaktifkan dan semua kredensial tersedia
ENABLE_POSTGRES_LOAD = os.getenv('ENABLE_POSTGRES_LOAD','0') == '1'
required_env = ['PGHOST','PGDATABASE','PGUSER','PGPASSWORD']
missing_env = [key for key in required_env if not os.getenv(key)]
if not ENABLE_POSTGRES_LOAD:
    print('PostgreSQL load NONAKTIF; Parquet sudah selesai.')
elif missing_env:
    print('Load dibatalkan. Variabel belum tersedia:', ', '.join(missing_env))
else:
    url = f"jdbc:postgresql://{os.environ['PGHOST']}:{os.getenv('PGPORT','5432')}/{os.environ['PGDATABASE']}"
    (df.write.format('jdbc').option('url',url).option('dbtable','umkm_lab.umkm_performance')
       .option('user',os.environ['PGUSER']).option('password',os.environ['PGPASSWORD'])
       .option('driver','org.postgresql.Driver').option('batchsize','5000')
       .mode('append').save())
    print('Load PostgreSQL selesai:', input_rows, 'baris')

## Checklist validasi akhir
- Jumlah input sama dengan output Parquet.
- Tidak ada ID duplikat.
- Tidak ada pelanggaran rentang nilai inti.
- PostgreSQL hanya diaktifkan dengan kredensial environment, bukan password di notebook.